In [ ]:
error_list = []

In [ ]:
import subprocess
import os

llvm_path = "/home/dxic/llvmlearn/llvm-project/llvm/build/bin/"   
def compile_to_ir(testcase):
    """将 testcase 编译为 LLVM IR 文本。

    Args:
        testcase: dict，包含 compiler, flags, input_file, id 等字段

    Returns:
        str: 编译后的 LLVM IR 文本
    """
    compiler = os.path.join(llvm_path, testcase["compiler"])
    input_file = testcase["input_file"]

    cmd = [compiler, "-S", "-emit-llvm", *testcase["flags"], "-o", "-", input_file]
    try:
        result = subprocess.run(cmd, check=True, stdout=subprocess.PIPE, text=True)
        # print(f"[OK] {testcase['id']} ({len(result.stdout)} chars)")
        return result.stdout
    except subprocess.CalledProcessError as e:
        # print(f"[FAIL] {testcase['id']}\nstderr: {e.stderr}")
        raise Exception(f"[FAIL] {testcase['id']} stderr: {e.stderr}")

 
if __name__ == "__main__":
    testcase = {
        "compiler": "clang",
        "flags" : [],
        "input_file": "/home/dxic/llvmtest/test/twofunc.c",
        "id": "twofunc"
    }
    testcase["flags"] += ["-O3",  "-Xclang", "-disable-llvm-passes",]
    ir = compile_to_ir(testcase)
    testcase["flags"] += ["-g"]
    ir_g = compile_to_ir(testcase)
    print(ir_g)

In [ ]:
import re


def parse_opt_stderr(stderr):
    """从 opt-pipe0 的 stderr 输出中解析出结构化统计。

    Args:
        stderr: opt-pipe0 输出的 stderr 文本

    Returns:
        list[dict]: 每个 pass 一条，包含 pass, ir, invalidated, funcs 字段
    """
    func_re = re.compile(
        r'^\s+(\S+)\s+L\((\d+)->(\d+)\)\s+BB\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'Inst\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'NBB\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'NI\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'(?:NEBB\(\+(\d+),-(\d+)\)\s*)?'
        r'(?:\s+\[DESTROYED\])?'
    )
    records = []
    cur = None
    for line in stderr.splitlines():
        if line.startswith('['):
            parts = line[1:].split(']', 1)
            rest = parts[1].strip() if len(parts) > 1 else ""
            invalidated = rest.endswith(' [INVALIDATED]')
            if invalidated:
                rest = rest[:-len(' [INVALIDATED]')]
            cur = {"pass": parts[0].strip(), "ir": rest, "invalidated": invalidated, "funcs": []}
            records.append(cur)
        elif cur:
            m = func_re.match(line)
            if m:
                g = m.groups()
                cur["funcs"].append({
                    "name": g[0],
                    "L_orig": int(g[1]), "L_new": int(g[2]),
                    "BB_orig": int(g[3]), "BB_add": int(g[4]),
                    "BB_del": int(g[5]), "BB_mod": int(g[6]),
                    "Inst_orig": int(g[7]), "Inst_add": int(g[8]),
                    "Inst_del": int(g[9]), "Inst_mod": int(g[10]),
                    "NBB_orig": int(g[11]), "NBB_add": int(g[12]),
                    "NBB_del": int(g[13]), "NBB_mod": int(g[14]),
                    "NI_orig": int(g[15]), "NI_add": int(g[16]),
                    "NI_del": int(g[17]), "NI_mod": int(g[18]),
                    "NEBB_add": int(g[19]) if g[19] is not None else 0,
                    "NEBB_del": int(g[20]) if g[20] is not None else 0,
                    "destroyed": "[DESTROYED]" in line,
                })
    return records


if __name__ == "__main__":
    records = parse_opt_stderr(stderr)
    print(f"Total passes: {len(records)}")
    for r in records:
        print(r)

In [ ]:
import re
from collections import defaultdict

O3_loopopt_segment_pattern ={
    "simLPM1": ("LoopSimplifyPass", 
                "SimplifyCFGPass"),
    "simLPM2": ("LoopSimplifyPass", 
                "SROAPass"),
    "simLPM3": ("LoopSimplifyPass", 
                "CoroElidePass"),
    "optLPM1": ("LoopSimplifyPass", 
                "LoopDistributePass"),
    "vectorize": ("LoopVectorizePass", 
                "InferAlignmentPass"),
    "loadelim": ("LoopLoadEliminationPass", 
                "InstCombinePass"),
    "unroll": ("LoopUnrollPass", 
                "WarnMissedTransformationsPass"),
    "optLPM2": ("LoopSimplifyPass", 
                "AlignmentFromAssumptionsPass"),
    "codegenLPM": ("LoopSimplifyPass", 
                "MergeICmpsPass"),
}

def group_pass_segments(records, pattern):
    """将 pass 记录按 pattern 分组为段。

    每个段由 (start_pass, end_pass) 定义，段内所有 pass 必须作用于同一个函数。

    Args:
        records: parse_opt_stderr 返回的记录列表
        pattern: {段名: (start_pass, end_pass)} 字典

    Returns:
        dict: {函数名: [(段名, 开始编号, 结束编号), ...]}
    """
    def get_funcs_from_ir(ir_str):
        """从 ir 字段提取涉及的函数名列表。"""
        if not ir_str:
            return []
        if ir_str.startswith("Module"):
            return []
        if ir_str.startswith("function:"):
            return [ir_str[len("function:"):]]
        if ir_str.startswith("loop:"):
            m = re.search(r'function:(\S+)', ir_str)
            return [m.group(1)] if m else []
        if ir_str.startswith("scc:"):
            return ir_str[len("scc:"):].split(",")
        if ir_str.startswith("machine:"):
            return [ir_str[len("machine:"):]]
        return []

    result = defaultdict(list)
    # pending[seg_name][func_name] = start_index
    pending = {seg: {} for seg in pattern}

    for i, rec in enumerate(records):
        pass_name = rec["pass"]
        funcs = get_funcs_from_ir(rec["ir"])

        for seg_name, (start_pass, end_pass) in pattern.items():
            if pass_name == start_pass:
                for fn in funcs:
                    if fn not in pending[seg_name]:
                        pending[seg_name][fn] = i
            elif pass_name == end_pass:
                for fn in funcs:
                    if fn in pending[seg_name]:
                        result[fn].append((seg_name, pending[seg_name][fn], i))
                        # 清空该函数所有段的 pending
                        for s in pending:
                            pending[s].pop(fn, None)

    return dict(result)

if __name__ == "__main__":
    func_seg = group_pass_segments(records, O3_loopopt_segment_pattern)
    print(*func_seg.items(), sep="\n")


In [ ]:
O3_pass_in_segments={
    "LoopInstSimplifyPass": ["simLPM1"],
    "LoopSimplifyCFGPass": ["simLPM1"],
    "LICMPass": ["simLPM1"], # ["simLPM1", "simLPM3", "optLPM2"],
    "LoopRotatePass": ["simLPM1"], # ["simLPM1", "optLPM1"],
    "SimpleLoopUnswitchPass": ["simLPM1"], # ["simLPM1", "simLPM2"],
    "LoopIdiomRecognizePass": ["simLPM2"],
    "IndVarSimplifyPass": ["simLPM2"],
    "LoopDeletionPass": ["simLPM2"], # ["simLPM2", "optLPM1"],
    "LoopFullUnrollPass": ["simLPM2"],

    "LoopVectorizePass": ["vectorize"],
    "LoopLoadEliminationPass": ["loadelim"],
    "LoopUnrollPass": ["unroll"],
    "LoopStrengthReducePass": ["codegenLPM"],
}
# licm依赖loop-mssa包装，没有mssa会报错，当licm位于-passes第一个时，自动使用loop-mssa包装
# loop-rotate和simple-loop-unswitch的优化行为不受mssa影响，只更新mssa信息
strategy_O3 = {
    ("vectorize", "LoopVectorizePass") : (
        ["-passes=loop(no-op-loop)"],
        ["-passes=loop-vectorize<no-interleave-forced-only;no-vectorize-forced-only;>"]),
    ("loadelim", "LoopLoadEliminationPass") : (
        ["-passes=loop(no-op-loop)"],
        ["-passes=loop-load-elim"]),
    ("unroll", "LoopUnrollPass"): (
        ["-passes=loop(no-op-loop)"],
        ["-passes=loop-unroll<O3>"]),
    ("simLPM1", "LoopRotatePass"): (
        ["-passes=licm<no-allowspeculation>"],
        ["-passes=loop-rotate<header-duplication;no-prepare-for-lto>"]),
    ("simLPM1", "SimpleLoopUnswitchPass") : (
        ["-passes=loop-mssa(loop-instsimplify,loop-simplifycfg,licm<no-allowspeculation>,loop-rotate<header-duplication;no-prepare-for-lto>,licm<allowspeculation>)"],
        ["-passes=simple-loop-unswitch<nontrivial;trivial>"]),
    ("simLPM2", "LoopFullUnrollPass"): (
        ["-passes=loop(loop-idiom,indvars,eslu<nontrivial;trivial>,loop-deletion)"],
        ["-passes=luf<O3>"]),
        

    ("simLPM1", "LICMPass"): (
        ["-passes=loop-rotate<header-duplication;no-prepare-for-lto>"],
        ["-passes=licm<allowspeculation>"],
        ),
    ("simLPM2", "LoopIdiomRecognizePass"):(
        ["-passes=loop(no-op-loop)"],
        ["-passes=loop-idiom"]),
    ("simLPM2", "IndVarSimplifyPass"): (
        ["-passes=loop(loop-idiom)"],
        ["-passes=indvars"]),
    ("codegenLPM", "LoopStrengthReducePass"): (
        ["-passes=loop(no-op-loop)"],
        ["-passes=loop-reduce"]),

    # may be used in future
    # ("simLPM2", "LoopDeletionPass"): (
    #     ["-passes=loop(loop-idiom,indvars,eslu<nontrivial;trivial>,)"],
    #     ["-passes=loop-deletion"]),
}


def find_cand_passes(records, segments, pass_in_segments):
    """找出真正修改了函数的 pass 及其所属 segment。

    Args:
        records: parse_opt_stderr 返回的记录列表
        segments: group_pass_segments 返回的 {函数名: [(段名, start, end), ...]}
        pass_in_segments: {pass名: [段名, ...]} 关注的 pass 到段的映射

    Returns:
        list[dict]: 每条记录包含 func, segment, pass, pass_index
    """
    def find_seg(fn, idx):
        """查找 idx 属于 fn 的哪个 segment。"""
        for i, (seg_name, start, end) in enumerate(segments.get(fn, [])):
            if start <= idx < end:
                return seg_name, i
        return None, None

    result = []
    for i, rec in enumerate(records):
        pass_name = rec["pass"]
        if pass_name not in pass_in_segments:
            continue
        for f in rec["funcs"]:
            if f["NI_add"] == 0 and f["NI_del"] == 0:
                continue
            fn = f["name"]
            seg_name, seg_index = find_seg(fn, i)
            if seg_name is None:
                continue
            if seg_name not in pass_in_segments[pass_name]:
                continue
            result.append({
                "func": fn,
                "segment": seg_name,
                "segment_index": seg_index,
                "pass": pass_name,
                "pass_index": i,
            })

    return result


if __name__ == "__main__":
    cand_list = find_cand_passes(records, func_seg, O3_pass_in_segments)
    # cand_list = filter_cand_list(cand_list, strategy_O3)
    print(*cand_list, sep="\n")


In [ ]:
def get_func_from_module(ir, funcname):
    """用 llvm-extract 从模块 IR 中提取指定函数。

    Args:
        ir: 模块 IR 文本
        funcname: 要提取的函数名

    Returns:
        str: 包含指定函数的模块 IR 文本
    """
    cmd_list = [
        llvm_path+"llvm-extract",
        "-func="+funcname,
        "-keep-const-init",
        "-S",
        "-o",
        "-",
    ]
    result = subprocess.run(cmd_list, check=True, input=ir, stdout=subprocess.PIPE, text=True)
    return result.stdout

if __name__ == "__main__":
    f_ir = get_func_from_module(ir_g, "f")
    print(f_ir)

In [ ]:
def get_opt_ir(ir, optarg):
    """对 IR 运行 opt 优化，返回优化后的 IR 文本。

    Args:
        ir: LLVM IR 文本
        optarg: opt 参数列表，如 ["-O3"] 或 ["-passes=licm"]

    Returns:
        str: 优化后的 IR 文本
    """
    cmd = [
        llvm_path + "/opt",
        *optarg,
        "-S",
    ]
    result = subprocess.run(cmd, input=ir, check=True,
                              stdout=subprocess.PIPE, text=True)
    return result.stdout

if __name__ == "__main__":
    if ir:
        ir_opt = get_opt_ir(ir, ["-O3"])
        print(ir_opt)


In [ ]:
irdiff_path = "/home/dxic/LoopOptDiff2/LoopOptDataset/build/ir-diff"

def diff_func_in_2_ir(func, ir0, ir1):
    """用 ir-diff 比较两个模块 IR 中同名函数的差异，返回解析后的字典。

    Args:
        func: 函数名
        ir0: 第一个模块 IR 文本（before）
        ir1: 第二个模块 IR 文本（after）

    Returns:
        dict: 包含 name, L_orig, L_new, BB_orig, BB_add, BB_del, BB_mod,
              Inst_orig, Inst_add, Inst_del, Inst_mod, NBB_*, NI_*, NEBB_*,
              destroyed 等字段
    """

    input = ir0 + '\n---\n' + ir1

    cmd = [irdiff_path, func]
    result = subprocess.run(cmd, input=input, check=True, stdout=subprocess.PIPE, text=True)

    line = result.stdout.strip()
    m = re.match(
        r'(\S+)\s+L\((\d+)->(\d+)\)\s+BB\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'Inst\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'NBB\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'NI\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'NEBB\(\+(\d+),-(\d+)\)'
        r'(\s+\[DESTROYED\])?',
        line
    )
    if not m:
        raise RuntimeError(f"ir-diff output parse failed: {line}")

    return {
        "name": m.group(1),
        "L_orig": int(m.group(2)), "L_new": int(m.group(3)),
        "BB_orig": int(m.group(4)), "BB_add": int(m.group(5)),
        "BB_del": int(m.group(6)), "BB_mod": int(m.group(7)),
        "Inst_orig": int(m.group(8)), "Inst_add": int(m.group(9)),
        "Inst_del": int(m.group(10)), "Inst_mod": int(m.group(11)),
        "NBB_orig": int(m.group(12)), "NBB_add": int(m.group(13)),
        "NBB_del": int(m.group(14)), "NBB_mod": int(m.group(15)),
        "NI_orig": int(m.group(16)), "NI_add": int(m.group(17)),
        "NI_del": int(m.group(18)), "NI_mod": int(m.group(19)),
        "NEBB_add": int(m.group(20)), "NEBB_del": int(m.group(21)),
        "destroyed": m.group(22) is not None,
    }

if __name__ == "__main__":
    print(diff_func_in_2_ir("f", ir, ir_g))

In [ ]:
import re

def available_externally_to_external(ir: str) -> str:
    """将 LLVM IR 中函数定义的 available_externally linkage 改为 external。"""
    return re.sub(
        r"(?m)^(\s*define\s+)available_externally\b",
        r"\1external",
        ir,
    )

if __name__ == "__main__":
    with open("./test/debug_ir.txt", "r") as f:
        debug_ir = f.read()
    debug_ir_x = available_externally_to_external(debug_ir)
    with open("./test/debug_ir_x.txt", "w") as f:
        f.write(debug_ir_x)


In [ ]:
import subprocess

def get_function_source_locations(ir):
    """调用 ir-source-loc，返回 {函数名: (源文件路径, 行号)}。"""
    tool = "/home/dxic/LoopOptDiff2/LoopOptDataset/build/ir-source-loc"
    result = subprocess.run(
        [tool],
        input=ir,
        text=True,
        stdout=subprocess.PIPE,
        check=True,
    )

    locations = {}
    for output_line in result.stdout.splitlines():
        function, path, line = output_line.rsplit("\t", 2)
        locations[function] = (None if path == "<unknown>" else path, int(line))
    return locations

if __name__ == "__main__":
    with open("./test/debug_ir.txt", "r") as f:
        debug_ir = f.read()
    locations = get_function_source_locations(debug_ir)
    print(*locations.items(), sep="\n")

In [ ]:
import re
import threading

def get_cand_ir2(ir, level, cand_list, seg_group):
    indexes = sorted({seg_group[c["func"]][c["segment_index"]][1] - 1 for c in cand_list})
    if not indexes: return
    proc = None; errors = []; thread = None; done = False; pending = bytearray()
    def read_frame():
        while True:
            i = pending.find(b"\x1e")
            if i >= 0: data = bytes(pending[:i]); del pending[:i + 1]; return data
            chunk = proc.stdout.read1(4096)
            if not chunk:
                if pending: raise RuntimeError("opt-pipe2 ended with an unterminated stdout frame")
                return None
            pending.extend(chunk)
    try:
        proc = subprocess.Popen(["./build/opt-pipe2", level, ",".join(map(str, indexes))], stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        thread = threading.Thread(target=lambda: errors.append(proc.stderr.read()), daemon=True); thread.start()
        try: proc.stdin.write(ir.encode() + b"\x1e"); proc.stdin.flush()
        except (BrokenPipeError, OSError) as exc: raise RuntimeError("opt-pipe2 stdin broken while sending initial IR") from exc
        for n, expected in enumerate(indexes):
            if n:
                try: proc.stdin.write(b"\x1e"); proc.stdin.flush()
                except (BrokenPipeError, OSError) as exc: raise RuntimeError("opt-pipe2 stdin broken while requesting frame") from exc
            data = read_frame()
            if data is None: raise RuntimeError(f"opt-pipe2 missing frame for pass #{expected}")
            try: text = data.decode("utf-8")
            except UnicodeDecodeError as exc: raise RuntimeError("opt-pipe2 emitted non-UTF-8 stdout") from exc
            m = re.match(r"\A; \*\*\* IR after pass #(\d+) [^\n]*\n", text)
            if not m: raise RuntimeError("opt-pipe2 frame has invalid or missing marker")
            actual = int(m.group(1))
            if actual != expected: raise RuntimeError(f"opt-pipe2 marker pass #{actual}, expected #{expected}")
            yield actual, text[m.end():]
        proc.stdin.close()
        while True:
            chunk = proc.stdout.read1(4096)
            if not chunk: break
            pending.extend(chunk)
        proc.wait(); thread.join()
        if pending: raise RuntimeError("opt-pipe2 emitted extra stdout frame data")
        done = True
    finally:
        if proc is not None and not done:
            try: proc.stdin.close()
            except (OSError, AttributeError): pass
            proc.terminate()
            try: proc.wait(timeout=5)
            except subprocess.TimeoutExpired: proc.kill(); proc.wait()
            if thread is not None: thread.join(timeout=5)

from pathlib import Path


def verify_cand_ir2(cand, module_ir, strategy):
    strategy_key = (cand["segment"], cand["pass"])
    if strategy_key not in strategy:
        return None

    module_ir = available_externally_to_external(module_ir)
    ir0 = get_func_from_module(module_ir, cand["func"])
    
    opt1, opt2 = strategy[strategy_key]
    ir1 = get_opt_ir(ir0, opt1)
    ir2 = get_opt_ir(ir1, opt2)

    ir1x = ir1
    ir2x = ir2
    # optx = ["-passes=simplifycfg,instsimplify"]
    # ir1x = get_opt_ir(ir1, optx)
    # ir2x = get_opt_ir(ir2, optx)

    diff = diff_func_in_2_ir(cand["func"], ir1x, ir2x)
    if diff["NI_add"] + diff["NI_del"] <= 0:
        return None
    return {"diff": diff, "ir1": ir1, "ir2": ir2, **cand}


def single_get2(
        testcase, 
        input_path, 
        output_path,
        level_config,
    ):
    """逐个提取 candidate 模块 IR，验证后立即输出函数 IR，并汇总结果。

    candidate 统计基于按 strategy 过滤后的 cand_list；返回 (results, stats)，
    stats 包含 candidate_functions 和 verified_functions 两个 {pass: 函数数} 映射。
    """
    level, pattern, pass_in_segments, strategy = level_config

    testcase["flags"] += ["-O3", "-Xclang", "-disable-llvm-passes"]
    stderr_path = Path(input_path) / (testcase["id"] + ".txt")
    with open(stderr_path, "r") as f:
        content = f.read()
    if "========summary========" in content:
        stderr = content[:content.index("========summary========")]
    else:
        stderr = content

    testcase["flags"] += ["-g"]
    ir_g = compile_to_ir(testcase)
    records = parse_opt_stderr(stderr)
    func_seg = group_pass_segments(records, pattern)
    cand_list = find_cand_passes(records, func_seg, pass_in_segments)
    cand_list = [
        cand for cand in cand_list 
        if (cand["segment"], cand["pass"]) in strategy
    ]
    candidate_functions = {}
    for cand in cand_list:
        candidate_functions.setdefault(cand["pass"], set()).add(cand["func"])
    loc = get_function_source_locations(ir_g)


    cand_by_pass_index = {}
    for cand in cand_list:
        pass_index = func_seg[cand["func"]][cand["segment_index"]][1] - 1
        cand_by_pass_index.setdefault(pass_index, []).append(cand)

    results = {}
    for pass_index, module_ir in get_cand_ir2(ir_g, level, cand_list, func_seg):
        if pass_index not in cand_by_pass_index: raise RuntimeError(f"opt-pipe2 returned unexpected pass #{pass_index}")
        current_cands = cand_by_pass_index[pass_index]
        for cand in current_cands:
            result = verify_cand_ir2(cand, module_ir, strategy)
            if result is None:
                continue
            result_key = (cand["func"], cand["pass"])
            func, passname = result_key
            diff = result["diff"]
            score = diff["NI_add"] + diff["NI_del"]
            old_result = results.get(result_key)
            if old_result is not None:
                old_score = old_result["NI_add"] + old_result["NI_del"]
                if score <= old_score:
                    continue
                old_result["ir1_path"].unlink(missing_ok=True)
                old_result["ir2_path"].unlink(missing_ok=True)

            if func in loc:
                result["loc"] = loc[func]
            else:
                loc_ir1 = get_function_source_locations(result["ir1"])
                result["loc"] = loc_ir1[func]

            result["id"] = testcase["id"]
            del result["pass_index"]
            diff = result.pop("diff")
            del diff["name"]
            result.update(diff)

            pass_dir = Path(output_path) / passname
            pass_dir.mkdir(parents=True, exist_ok=True)
            result["ir1_path"] = (pass_dir / (testcase["id"] + "-" + func + "-1.ll")).resolve()
            result["ir2_path"] = (pass_dir / (testcase["id"] + "-" + func + "-2.ll")).resolve()

            with open(result["ir1_path"], "w") as f:
                f.write(f"; loc: {result['loc'][0]}:{result['loc'][1]}\n")
                f.write(f"; id: {testcase['id']}\n")
                f.write(f"; func: {func}\n")
                f.write(f"; pass: {passname}\n")
                f.write(f"; segment: {result['segment']},{result['segment_index']}\n")
                if "L_orig" in result:
                    f.write(f"; L: {result['L_orig']} -> {result['L_new']}\n")
                    f.write(f"; BB: {result['BB_orig']} +{result['BB_add']} -{result['BB_del']} ~{result['BB_mod']}\n")
                    f.write(f"; Inst: {result['Inst_orig']} +{result['Inst_add']} -{result['Inst_del']} ~{result['Inst_mod']}\n")
                    f.write(f"; NBB: {result['NBB_orig']} +{result['NBB_add']} -{result['NBB_del']} ~{result['NBB_mod']}\n")
                    f.write(f"; NI: {result['NI_orig']} +{result['NI_add']} -{result['NI_del']} ~{result['NI_mod']}\n")
                    f.write(f"; NEBB: +{result['NEBB_add']} -{result['NEBB_del']}\n")
                    f.write(f"; destroyed: {result['destroyed']}\n")
                    f.write("\n")
                f.write(result["ir1"])
            del result["ir1"]

            with open(result["ir2_path"], "w") as f:
                f.write(result["ir2"])
            del result["ir2"]

            results[result_key] = result

    verified_functions = {}
    for func, passname in results:
        verified_functions.setdefault(passname, set()).add(func)

    pass_names = sorted(candidate_functions)
    stats = {
        "candidate_functions": {
            passname: len(candidate_functions[passname])
            for passname in pass_names
        },
        "verified_functions": {
            passname: len(verified_functions.get(passname, ()))
            for passname in pass_names
        },
    }
    return results, stats

if __name__ == "__main__":
    O3_config = ("-O3", O3_loopopt_segment_pattern, O3_pass_in_segments, strategy_O3)
    results, stats = single_get2(testcase, "./test/", "./test/getobj", O3_config)
    print(*results.items(), sep="\n")
    print(stats)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed



# 排除pass，loc相同的例子
def batch_get2(testcases, input_path, output_path,  level_config,n_workers):
    """并行处理多个 testcase，并按 (pass, loc) 聚合结果。

    Args:
        testcases: testcase dict 列表
        input_path: 预生成的 stderr 文件目录
        output_path: IR 文件输出目录
        level_config: (level, pattern, pass_in_segments, strategy)
        n_workers: 并行线程数

    Returns:
        tuple[dict, dict]: (all_results, stats)。all_results 为每个 (pass, loc)
        保留 NI 增删数最多的 result；stats 包含 candidate_functions、
        verified_functions 和 source_unique_functions 三个 {pass: 函数数} 映射，
        pass 键统一并用 0 补齐。candidate_functions/verified_functions 只汇总
        成功返回的 testcase；source_unique_functions 按最终 (pass, loc) 去重统计。
    """
    all_results = {}
    candidate_stats = {}
    verified_stats = {}
    total = len(testcases)
    done = 0
    global error_list

    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = {
            executor.submit(single_get2, 
                            tc,  input_path, output_path, level_config): tc
            for tc in testcases
        }

        for future in as_completed(futures):
            tc = futures[future]
            done += 1
            try:
                result, testcase_stats = future.result()
                for r in result.values():
                    key = (r["pass"], r["loc"])
                    old_result = all_results.get(key)
                    if old_result is None:
                        all_results[key] = r
                        continue

                    old_score = (old_result["NI_add"] + old_result["NI_del"])
                    new_score = (r["NI_add"] + r["NI_del"])
                    if new_score > old_score:
                        if output_path is not None:
                            old_result["ir1_path"].unlink()
                            old_result["ir2_path"].unlink()

                        print(
                            f"WARNING: duplicate key {key}, replacing result "
                            f"because (NI_add + NI_del) increased from "
                            f"{old_score:.6g} to {new_score:.6g}"
                        )
                        all_results[key] = r
                    elif output_path is not None:
                        r["ir1_path"].unlink()
                        r["ir2_path"].unlink()

                for passname, count in testcase_stats["candidate_functions"].items():
                    candidate_stats[passname] = candidate_stats.get(passname, 0) + count
                for passname, count in testcase_stats["verified_functions"].items():
                    verified_stats[passname] = verified_stats.get(passname, 0) + count
                print(f"[{done}/{total}] {tc['id']} OK")
            except Exception as e:
                print(f"[{done}/{total}] {tc['id']} FAIL: {e}")
                error_list.append((tc['id'],e))

    source_unique_stats = {}
    for passname, _loc in all_results:
        source_unique_stats[passname] = source_unique_stats.get(passname, 0) + 1
    pass_names = sorted(set(candidate_stats) | set(verified_stats) | set(source_unique_stats))
    stats = {
        "candidate_functions": {
            passname: candidate_stats.get(passname, 0)
            for passname in pass_names
        },
        "verified_functions": {
            passname: verified_stats.get(passname, 0)
            for passname in pass_names
        },
        "source_unique_functions": {
            passname: source_unique_stats.get(passname, 0)
            for passname in pass_names
        },
    }
    return all_results, stats

if __name__ == "__main__":
    import json
    with open("test/testsuitex.json", "r") as f:
        testsuite = json.load(f)
    O3_config = ("-O3", O3_loopopt_segment_pattern, O3_pass_in_segments, strategy_O3)
    r, stats = batch_get2(testsuite[:3], "../RQ1/test/260715/O3", "./test/getobj" , O3_config, 8,)
    print(*r.values(), sep="\n")
    print(stats)
    del testsuite

In [ ]:
if __name__ == "__main__":

    import json
    with open("/home/dxic/LoopOptDiff2/extract/testsuite260808.json", "r") as f:
        testsuite = json.load(f)
    with open("/home/dxic/LoopOptDiff2/extract/coreutils260808.json", "r") as f:
        coreutils = json.load(f)
    with open("/home/dxic/LoopOptDiff2/extract/openssl260808.json", "r") as f:
        openssl = json.load(f)
    with open("/home/dxic/LoopOptDiff2/extract/ffmpeg260808.json", "r") as f:
        ffmpeg = json.load(f)
    # with open("/home/dxic/LoopOptDiff2/extract/curl260808.json", "r") as f:
    #     curl = json.load(f)
    # with open("/home/dxic/LoopOptDiff2/extract/zlib260808.json", "r") as f:
    #     zlib = json.load(f)
    # with open("/home/dxic/LoopOptDiff2/extract/sqlite260808.json", "r") as f:
    #     sqlite = json.load(f)

    strategy_loopdeletion ={
        ("simLPM2", "LoopDeletionPass"): (
            ["-passes=loop(loop-idiom,indvars,eslu<nontrivial;trivial>)"],
            ["-passes=loop-deletion"]),
    }
    O3_loopdeletion_config = (
        "-O3", 
        O3_loopopt_segment_pattern, 
        O3_pass_in_segments, 
        strategy_loopdeletion)


    input_path = "../RQ1/test/260811/O3"
    output_path = "./test/261021/ll"
    tcs = testsuite + coreutils + openssl + ffmpeg
    error_list = []
    rO3, statsO3 = batch_get2(tcs, input_path, output_path, O3_loopdeletion_config, 16)

    print("error_list:", len(error_list))
    print(*error_list, sep="\n")